# 06 - Bronze: IMF WEO fiscal and macro context

This notebook ingests IMF World Economic Outlook JSONL produced by the local extractor and writes `bronze.imf_weo_raw`.

Local extraction:

```bash
python3 extraction/extract/imf_weo_extract.py \
  --all-cemac-ecowas \
  --start-year 1990 --end-year 2024 \
  --out data/raw/weo/cemac_ecowas_weo_1990_2024.jsonl
```

Upload to Databricks:

```bash
databricks fs mkdirs dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/weo -p cemac-project
databricks fs cp data/raw/weo/cemac_ecowas_weo_1990_2024.jsonl \
  dbfs:/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/weo/cemac_ecowas_weo_1990_2024.jsonl \
  --overwrite -p cemac-project
```

Indicators include debt-to-GDP, government revenue, expenditure, fiscal balance, current account balance, GDP growth, CPI inflation, and GDP in current USD.

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("USE SCHEMA bronze")

VOLUME_PATH = "/Volumes/cemac_ecowas_aes_trade/bronze/raw_landing/weo/"
loaded_at = datetime.now(timezone.utc)

print("Catalog: cemac_ecowas_aes_trade")
print("Schema: bronze")
print(f"Volume path: {VOLUME_PATH}")

In [ ]:
# Cell 2 - List uploaded JSONL files
files = [f for f in dbutils.fs.ls(VOLUME_PATH) if f.name.endswith(".jsonl")]

if not files:
    raise FileNotFoundError(
        f"No .jsonl files found at {VOLUME_PATH}. Run imf_weo_extract.py locally and upload the output first."
    )

print(f"Found {len(files)} JSONL file(s):")
for file in files:
    print(f"  {file.name} ({file.size:,} bytes)")

In [ ]:
# Cell 3 - Read JSONL and normalize schema
paths = [file.path for file in files]
raw_df = spark.read.json(paths)

df = raw_df.select(
    F.col("source").cast("string").alias("source"),
    F.col("dataset").cast("string").alias("dataset"),
    F.col("frequency").cast("string").alias("frequency"),
    F.col("country_iso3").cast("string").alias("country_iso3"),
    F.col("country_name").cast("string").alias("country_name"),
    F.col("indicator_code").cast("string").alias("indicator_code"),
    F.col("indicator_name").cast("string").alias("indicator_name"),
    F.col("topic").cast("string").alias("topic"),
    F.col("unit").cast("string").alias("unit"),
    F.col("year").cast("int").alias("year"),
    F.col("value").cast("double").alias("value"),
    F.col("scale").cast("string").alias("scale"),
    F.col("decimals_displayed").cast("string").alias("decimals_displayed"),
    F.col("country_update_date").cast("string").alias("country_update_date"),
    F.col("overlap").cast("string").alias("overlap"),
    F.col("extracted_at").cast("string").alias("extracted_at"),
).dropDuplicates(["country_iso3", "indicator_code", "year"])

df = df.withColumn("loaded_at", F.lit(loaded_at))

print(f"Rows: {df.count():,}")
df.printSchema()
df.show(10, truncate=False)

In [ ]:
# Cell 4 - Write bronze.imf_weo_raw
spark.sql("DROP TABLE IF EXISTS bronze.imf_weo_raw")

(df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.imf_weo_raw"))

print("Write complete: bronze.imf_weo_raw")

In [ ]:
# Cell 5 - Verification and sanity checks
print("Overall coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        COUNT(DISTINCT indicator_code) AS indicators,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.imf_weo_raw
""").show()

print("Per-indicator coverage:")
spark.sql("""
    SELECT
        indicator_code,
        indicator_name,
        unit,
        COUNT(*) AS rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        COUNT(value) AS non_null_values
    FROM bronze.imf_weo_raw
    GROUP BY indicator_code, indicator_name, unit
    ORDER BY indicator_code
""").show(truncate=False)

print("2024 debt-to-GDP snapshot:")
spark.sql("""
    SELECT
        country_iso3,
        country_name,
        ROUND(value, 1) AS debt_to_gdp_pct
    FROM bronze.imf_weo_raw
    WHERE indicator_code = 'GGXWDG_NGDP'
      AND year = 2024
    ORDER BY value DESC
""").show(25, truncate=False)

print("Fiscal balance in 2024:")
spark.sql("""
    SELECT
        country_iso3,
        country_name,
        ROUND(value, 1) AS net_lending_borrowing_pct_gdp
    FROM bronze.imf_weo_raw
    WHERE indicator_code = 'GGXCNL_NGDP'
      AND year = 2024
    ORDER BY value ASC
""").show(25, truncate=False)